<a href="https://colab.research.google.com/github/souro26/bayesian-a-b-testing/blob/main/examples/04_gaming_matchmaking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import argonx  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/souro26/bayesian-a-b-testing.git

# Example 04 — Gaming Matchmaking & The 'Winner-Takes-All' Decision

A mobile gaming company is testing three different matchmaking algorithms to increase player engagement.

### The Challenge of Multi-variant Testing
When you test 3 or more variants at once, traditional 'pairwise' tests (A vs B, A vs C, B vs C) become dangerous. 
1.  **P-value Inflation:** Every time you run a test, there's a 5% chance of a false positive. If you run 3 tests, that risk balloons to ~14%. 
2.  **The 'Best' Question:** Knowing that B is better than A isn't enough. You need to know if B is the *absolute best* among all options before you commit to a global rollout.

### The Bayesian Solution: Simultaneous Argmax
In a Bayesian framework, we don't do pairwise tests. We model all three variants simultaneously and compute the probability that each one is the **global maximum**. This accounts for the uncertainty of all variants at once, naturally 'correcting' for multiple comparisons without needing complex Bonferroni adjustments.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from argonx import Experiment

plt.style.use("seaborn-v0_8-muted")
sns.set_theme(style="whitegrid")

## 1. Simulating Engagement (Counts)

Engagement is measured by 'Sessions Played per User'—a count metric that follows a **Poisson** distribution.

*   **Control:** Average 3.2 sessions/day.
*   **V1 (Latency-Focused):** Slight improvement (3.4 sessions).
*   **V2 (Skill-Focused):** High-risk, high-reward (3.8 sessions).

In [ ]:
np.random.seed(42)
N = 1200

control_games = np.random.poisson(3.2, N)
variant_1_games = np.random.poisson(3.4, N)
variant_2_games = np.random.poisson(3.8, N)

df = pd.DataFrame(
    {
        "algorithm": ["control"] * N + ["latency_focused"] * N + ["skill_focused"] * N,
        "sessions": np.concatenate([control_games, variant_1_games, variant_2_games]),
    }
)

plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x="sessions", hue="algorithm", fill=True, palette="viridis")
plt.title("Raw Player Engagement Distributions", fontsize=14, fontweight="bold")
plt.xlabel("Sessions Played per Day", fontsize=12)
plt.show()

## 2. Running the 3-Way Experiment

We use the `poisson` model. `argonx` will compute the probability of each variant being the global winner.

In [ ]:
exp = Experiment(
    data=df,
    variant_col="algorithm",
    primary_metric="sessions",
    model="poisson",
    control="control",
)

result = exp.run()
result.summary()

## 3. Simultaneous Visualization

In [ ]:
result.plot(metric_name="Sessions per User")

### Conclusion

Notice the **Probability of Being Best** chart in the dashboard:
*   While `latency_focused` is better than control, its probability of being the *best* is extremely low because `skill_focused` exists.
*   In a traditional pairwise system, you might have ended up rolling out the latency variant simply because its test finished first or its p-value was smaller.
*   **argonx** provides a single, unified decision: ship `skill_focused`. It handles the 'Winner-Takes-All' logic correctly by considering all evidence at once.